# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates step-by-step how to explore and process the FAIR² dataset using the `mlcroissant` library, strictly following the Croissant metadata's `@id` structure for referencing dataset entities.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load dataset metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id` identifiers.

In [ ]:
# List record sets and their fields/columns by @id
print("Record Sets (by @id):")
for record_set in metadata.record_sets:
    print(f"  - {record_set['@id']}: {record_set['name'] if 'name' in record_set else 'no name'}")

# For each record set, list fields (by @id)
for record_set in metadata.record_sets:
    print(f"\nFields in record set '{record_set['@id']}':")
    for field in record_set['fields']:
        field_label = field.get('name', 'no name')
        print(f"    - {field['@id']}: {field_label}")
    # Show columns (for file-based record sets)
    if 'columns' in record_set:
        print(f"  Columns:")
        for col in record_set['columns']:
            col_label = col.get('name', 'no name')
            print(f"    - {col['@id']}: {col_label}")

## 3. Data Extraction
Load data from all record sets into DataFrames for analysis. All IDs are referenced via their `@id`.

In [ ]:
# Prepare a list of available record set @id's
record_set_ids = [r['@id'] for r in metadata.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    try:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records for record set: {record_set_id}")
    except Exception as e:
        print(f"Failed to load DataFrame for {record_set_id}:", e)

# Display columns of first record set as an example
if record_set_ids:
    main_record_set_id = record_set_ids[0]
    print(f"\nColumns in '{main_record_set_id}': {dataframes[main_record_set_id].columns.tolist()}")
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps such as filtering, normalizing, and grouping, referencing all fields strictly by their `@id`.

In [ ]:
# For demonstration, pick a record set and a numeric field (by @id)
# (Replace with an actual @id after examining the field list)

# Example: use the first available numeric column from the first record set
df = dataframes[main_record_set_id]

# Find a likely numeric field by checking dtypes
numeric_field_candidates = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
print(f"Numeric fields in '{main_record_set_id}': {numeric_field_candidates}")

if numeric_field_candidates:
    numeric_field_id = numeric_field_candidates[0]  # Use the first numeric field
    threshold = df[numeric_field_id].mean()  # Example dynamic threshold

    # Filter rows
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
    display(filtered_df.head())

    # Normalize
    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized '{numeric_field_id}' for filtered records:")
    display(filtered_df[[numeric_field_id, norm_col]].head())

    # Try to group by a possible categorical field
    group_field_candidates = [col for col in df.columns if df[col].dtype == 'object' and col != numeric_field_id]
    if group_field_candidates:
        group_field_id = group_field_candidates[0]
        grouped = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"\nGrouped data by '{group_field_id}':")
        display(grouped.head())
else:
    print("No numeric fields found in this record set for EDA.")

## 5. Visualization
Visualize the distribution or relationship between fields. All references are via `@id`.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_candidates:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()

    if group_field_candidates:
        plt.figure(figsize=(10,5))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
In this notebook, we've loaded the FAIR² dataset using the Croissant metadata schema, inspected record sets and fields by their `@id`, extracted and analyzed data in accordance with FAIR principles, and visualized selected features. All entities are referenced with their full Croissant `@id`—ensuring clarity and data provenance for reproducible research.

Next steps may include hypothesis testing, integration with other FAIR datasets, or domain-specific analysis.
